我们终于结束了第一章对于一些基础内容的探讨，接下来我们着重讨论随机变量上的问题，并将其用于概率论与数理统计方面的问题上。

那么，首先我们面对的问题就是如何生成大量的随机数据。

In [ ]:
# 采用均匀分布生成随机数，默认区间 0-1s
runif(10)
# 指定区间上下界为2，10
runif(10, 2, 10)
# 设置随机数生成种子，确保结果可重复
set.seed(222)
runif(10)


# 从一些常见的随机分布中生成随机样本，如二项，指数和正态等
rbinom(10, size = 1, prob = 0.5)
rexp(10, rate = 0.2)
rnorm(10, mean = 0, sd = 0.5)


# sample 是常用的抽样函数
# 指定对象为0和1，抽10次，有放回
sample(0:1, size = 10, replace = TRUE)
# 在对象1-3中抽取，并指定对应抽取概率
x <- sample(1:3, size = 100, replace = TRUE, prob = c(0.2, 0.3, 0.5))
print(x)
# 频数表
table(x)
# 频率表
prop.table(table(x))

# 打乱小写字母表，得到一个随机排列的小写字母
sample(letters)

# 从数据集中随机抽取样本
print(head(airquality))
subset <- sample(1:nrow(airquality), size = 10)
mysample <- airquality[subset, ]
print(mysample)

上面我们是使用了 ```sample``` 直接进行随机抽样，除此之外，我们还可以使用逆变换法进行抽样。这里，我们需要先对逆变换法做一个简短的介绍：

逆变换法生成随机样本的核心思想如下：

- 先生成均匀分布随机数 $U \sim {\rm Unif}(0, 1)$
- 再利用目标分布的分布函数反函数，把 $U$ 变成目标分布的样本 $X$，也就是说
  $$ X=F^{-1}(U) $$

In [ ]:
# 结合逆变换法的核心思想，我们的代码可以这样写
n <- 1000
u <- runif(n)
# 生成连续型随机样本 f(x) = 3x^2
x <- u^(1 / 3)
# 使用画图查看
jpeg("fig1.jpeg", height = 800, width = 800, quality = 100)
hist(x, prob = TRUE, main = expression(f(x) == 3 * x^2))
x <- seq(0, 1, 0.01)
lines(x, 3 * x^2)
dev.off()

# 生成指数分布
myrexp <- function(n, lambda) {
    u <- runif(n)
    x <- -log(u) / lambda
    return(x)
}
x <- myrexp(10, lambda = 1)
print(x)

# 生成二项分布
n <- 1000
p <- 0.4
u <- runif(n)
x <- as.integer(u > 1 - p)
# 估计样本方差和均值，判断生成效果
mean(x)
var(x)


# 生成几何分布
n <- 10
p <- 0.4
u <- runif(n)

# 方式1，向上取整
k1 <- ceiling(log(1 - u) / log(1 - p))
# 方式2，向下取整，u 和 1-u 是同分布的，所以两种写法差不多
k2 <- floor(log(u) / log(1 - p)) + 1
print(k1)
print(k2)


接下来，我们讨论一个比较复杂的逆变换法生成样本，生成对数分布的样本

In [ ]:
rlogarithmic <- function(n, theta) {
    u <- runif(n)

    N <- ceiling(-16 / log10(theta)) # 初始化 CDF 长度，因为对数分布可以取到无穷大
    k <- 1:N # 构造取值点
    a <- -1 / log(1 - theta) # 归一化常数
    fk <- exp(log(a) + k * log(theta) - log(k)) # 计算对应概率密度，使用 log 避免出现精度问题
    Fk <- cumsum(fk) # 累加求和计算概率分布

    x <- integer(n)
    for (i in 1:n)
    {
        x[i] <- as.integer(sum(u[i] > Fk))

        while (x[i] == N) # 如果 x == N，我们需要扩展 CDF 的长度
        {
            logf <- log(a) + (N + 1) * log(theta) - log(N + 1)
            fk <- c(fk, exp(logf))
            Fk <- c(Fk, Fk[N] + fk[N + 1])
            N <- N + 1
            x[i] <- as.integer(sum(u[i] > Fk))
        } # 一般不太会，因为 N 取得很大了
    }
    return(x + 1)
}

n <- 1000
theta <- 0.5 # 参数设置为0.5
x <- rlogarithmic(n, theta)
# 计算理论概率并与样本频率比较
k <- sort(unique(x))
p <- -1 / log(1 - theta) * theta^k / k
se <- sqrt(p * (1 - p) / n)

print(round(rbind(table(x) / n, p, se), 3))


前面我们学习了两种方法， ```sample``` 和逆变换法来生成随机样本，这里我们继续学习使用第三种方法：**接受-拒绝法**生成随机样本。

In [ ]:
n <- 1000
k <- 0 # 目前已经接受的样本个数
j <- 0 # 尝试的次数
y <- numeric(n)

while (k < n) {
    u <- runif(1)
    j <- j + 1
    x <- runif(1)
    if (x * (1 - x) > u) {
        k <- k + 1
        y[k] <- x
    }
}

print(j)

# 比较经验分位数和理论分位数
p <- seq(.1, .9, .1)
Qhat <- quantile(y, p) # 样本分位数
Q <- qbeta(p, 2, 2) # 理论分位数
se <- sqrt(p * (1 - p) / (n * dbeta(Q, 2, 2)^2))
print(round(rbind(Qhat, Q, se), 3))

下面，我们介绍一种巧妙的构造方法求解对数分布。这个对数分布的生成方法是经过严格证明的，而非是单纯的数值拟合，感兴趣的可以自行求证。

In [ ]:
# 使用变换法生成对数分布
n <- 1000
theta <- 0.5
u <- runif(n)
v <- runif(n)
x <- floor(1 + log(v) / log(1 - (1 - theta)^u))

k <- 1:max(x)
p <- -1 / log(1 - theta) * theta^k / k
se <- sqrt(p * (1 - p) / n)
p.hat <- tabulate(x) / n
print(round(rbind(p.hat, p, se), 3))

# 封装为函数，比之前的生成更简单
rlogarithmic2 <- function(n, theta) {
    stopifnot(all(theta > 0 & theta < 1))
    th <- rep(theta, length = n)
    u <- runif(n)
    v <- runif(n)
    x <- floor(1 + log(v) / log(1 - (1 - th)^u))
    return(x)
}

print(rlogarithmic2(10, 0.5))

在概率统计中，还有许多由正态分布构造出来的一些分布，比如卡方分布等等。除此之外，还有一些其他的特殊分布，针对这些，我们也需要考虑如何生成。

In [ ]:
# 生成卡方分布
n <- 10
nu <- 2
X <- matrix(rnorm(n * nu), n, nu)^2
# 对每行求和，方法1
y <- rowSums(X)
# 对每行使用sum，方法2
y <- apply(X, 1, sum)
print(y)

# 生成 Gamma 混合分布
# 用随机模拟的方法生成混合样本，并画出混合后的密度曲线
n <- 5000
k <- sample(1:5, size = n, replace = TRUE, prob = (1:5) / 15)
rate <- 1 / k
x <- rgamma(n, shape = 3, rate = rate)

# 画出混合密度的图
jpeg("fig2.jpeg", height = 800, width = 800, quality = 100)
plot(density(x), xlim = c(0, 40), ylim = c(0, .3), lwd = 3, xlab = "x", main = "")
for (i in 1:5) lines(density(rgamma(n, 3, 1 / i)))
dev.off()

# 直接用混合密度公式计算理论密度，并画图
f <- function(x, lambda, theta) {
    fx <- sum(dgamma(x, 3, lambda) * theta) # 在点x处的概率密度
    return(fx)
}

p <- c(.1, .2, .2, .3, .2)
lambda <- c(1, 1.5, 2, 2.5, 3)

x <- seq(0, 8, length = 200)
dim(x) <- length(x) # 方便apply处理
y <- apply(x, 1, f, lambda = lambda, theta = p)

# plot the density of the mixture
jpeg("fig3.jpeg", height = 800, width = 800, quality = 100)
plot(x, y, type = "l", ylim = c(0, .85), lwd = 3, ylab = "Density")

for (j in 1:5) {
    # 叠加每个单独 Gamma 成分的理论密度
    y <- apply(x, 1, dgamma, shape = 3, rate = lambda[j])
    lines(x, y)
}
dev.off()


让我们回顾一下之前的代码在做什么，我们讨论了一些随机变量的样本生成，从离散型到连续型。但是我们能发现一点，我们之前讨论的都是一元的随机变量问题，很自然的，我们要将其推广到关于多元随机变量的样本生成。因为多元的问题往往比较复杂，所以这里我们只讨论多元正态分布的随机变量。

总的来说，我们按照如下的方法来生成 $d$ 维多元正态分布 $X\sim N_d(\mu, \Sigma)$ ：
- 首先，我们生成 $d$ 维的标准正态分布 $Z\sim N_d(0,I)$ ，其中均值为 $0$ ，协方差矩阵为单位阵 $I$
- 接下来，我们需要找到一个矩阵 $R$ ，使其满足 $R^TR=\Sigma$ 或 $RR^T=\Sigma$
- 最后，我们只需令 $X=ZR+\mu$ ，那么就有 $X\sim N_d(\mu, \Sigma)$ 

In [ ]:
# 使用谱分解获取多维正态分布
rmvn.eigen <- function(n, mu, Sigma) {
    d <- length(mu) # 获取维数
    ev <- eigen(Sigma, symmetric = TRUE) # 进行特征分解
    lambda <- ev$values
    V <- ev$vectors
    R <- V %*% diag(sqrt(lambda)) %*% t(V)

    Z <- matrix(rnorm(n * d), nrow = n, ncol = d)
    X <- Z %*% R + matrix(mu, n, d, byrow = TRUE)
    return(X)
}

# 生成样本检验一下函数
# 定义均值和协方差矩阵
n <- 1000
mu <- c(0, 0)
Sigma <- matrix(c(1, 0.9, 0.9, 1), nrow = 2, ncol = 2)

X <- rmvn.eigen(n, mu, Sigma)
print(apply(X, 2, mean))
print(cor(X))

jpeg("fig4.jpeg", height = 800, width = 800, quality = 100)
plot(X, xlab = "x", ylab = "y", pch = 20)
dev.off()


In [ ]:
# 使用奇异值分解获取多维正态分布
rmvn.svd <- function(n, mu, Sigma) {
    d <- length(mu)
    S <- svd(Sigma) # 奇异值分解
    R <- S$u %*% diag(sqrt(S$d)) %*% t(S$v)

    Z <- matrix(rnorm(n * d), nrow = n, ncol = d)
    X <- Z %*% R + matrix(mu, n, d, byrow = TRUE)
    return(X)
}

# generate the sample to test the function
# generate mean and covariance parameters
n <- 1000
mu <- c(0, 0)
Sigma <- matrix(c(1, 0.9, 0.9, 1), nrow = 2, ncol = 2)

X <- rmvn.svd(n, mu, Sigma)
print(apply(X, 2, mean))
print(cor(X))

jpeg("fig5.jpeg", height = 800, width = 800, quality = 100)
plot(X, xlab = "x", ylab = "y", pch = 20)
dev.off()

In [ ]:
# 使用 Cholesky 分解获取多元正态分布
rmvn.Choleski <- function(n, mu, Sigma) {
    d <- length(mu)
    Q <- chol(Sigma) # Choleski factorization 返回一个上三角矩阵

    Z <- matrix(rnorm(n * d), nrow = n, ncol = d)
    X <- Z %*% Q + matrix(mu, n, d, byrow = TRUE)
    return(X)
}

# generating the samples according to the mean and covariance
# structure as the four-dimensional iris virginica data
y <- subset(x = iris, Species == "virginica")[, 1:4]
mu <- apply(y, 2, mean)
Sigma <- cov(y)
print(mu)
print(Sigma)

# now generate MVN data with this mean and covariance
X <- rmvn.Choleski(200, mu, Sigma)

jpeg("fig6.jpeg", height = 800, width = 800, quality = 100)
pairs(X)
dev.off()

关于生成多元正态分布的三种主要方法就是这样，接下来我们考虑对这几种方法进行性能评估。

In [ ]:
library(mvtnorm) # use the library to generate mvn sample

n <- 100 # sample size
d <- 30 # dimension
N <- 2000 # test N times

set.seed(10)
mu <- numeric(d) # mean
Sigma <- cov(matrix(rnorm(n * d), n, d)) # covariance

set.seed(100)
system.time(for (i in 1:N) rmvn.eigen(n, mu, Sigma))

set.seed(100)
system.time(for (i in 1:N) rmvn.svd(n, mu, Sigma))

set.seed(100)
system.time(for (i in 1:N) rmvn.Choleski(n, mu, Sigma))

set.seed(100)
system.time(for (i in 1:N) rmvnorm(n, mu, Sigma, method = "eigen"))

set.seed(100)
system.time(for (i in 1:N) rmvnorm(n, mu, Sigma, method = "svd"))

set.seed(100)
system.time(for (i in 1:N) rmvnorm(n, mu, Sigma, method = "chol"))


通过上面的测试，我们显然可以看出：使用 **Cholesky** 法是最快的，**eigen** 和 **svd** 相对更慢。但是，使用 **Cholesky** 要求协方差矩阵 $\Sigma$ 是正定的，这在数值上要求比较严格。而另外两种方法相对来说更加稳定和通用，具体使用时可以先做矩阵正定的判定。

多元的正态分布我们介绍完了，接下来我们考虑多元混合正态分布的样本生成。

PS：由于 `MASS` 包在 R 语言的官方仓库中没有，无法直接通过命令安装。所以我采用的是 `mvtnorm` 包中的函数进行生成。

In [ ]:
# 概括地讲，每一个样本都独立地先选“类别”，再从对应类别分布中生成。
loc.mix.0 <- function(n, p, mu1, mu2, Sigma) {
    # generate sample from BVN location mixture
    X <- matrix(0, n, length(mu1))

    for (i in 1:n)
    {
        # 使用伯努利试验控制采样的正态分布
        k <- rbinom(1, size = 1, prob = p)
        if (k) {
            X[i, ] <- rmvnorm(1, mu1, Sigma)
        } else {
            X[i, ] <- rmvnorm(1, mu2, Sigma)
        }
    }
    return(X)
}
x <- loc.mix.0(1000, 0.5, rep(0, 4), 2:5, Sigma = diag(4))
print(head(x))

根据第一章我们对 R 语言的初步介绍，我们可以知道：R 中的循环是比较慢的，所以上面的这个方法运行效率并不算高。为了解决这个问题，我们引入了改进后更高效的版本。

In [ ]:
# 先计算处所需要的样本数，然后一次性生成，接着混合样本，最后打乱
loc.mix <- function(n, p, mu1, mu2, Sigma) {
    # generate sample from BVN location mixture, version 2
    n1 <- rbinom(1, size = n, prob = p)
    n2 <- n - n1
    x1 <- rmvnorm(n1, mu1, Sigma)
    x2 <- rmvnorm(n2, mu2, Sigma)
    X <- rbind(x1, x2) # combine the samples
    return(X[sample(1:n), ]) # mix them
}
x <- loc.mix(1000, 0.5, rep(0, 4), 2:5, Sigma = diag(4))
print(head(x))

jpeg("fig7.jpeg", height = 800, width = 800, quality = 100)
r <- range(x) * 1.2
par(mfrow = c(2, 2))
for (i in 1:4) {
    hist(x[, i],
        xlim = r, ylim = c(0, .3), freq = FALSE,
        main = "", breaks = seq(-5, 10, 0.5)
    )
}
dev.off()